# Детекция ботов по событиям внутри суточного окна

Задача: по событиям куки внутри окна `[window_start_ts, window_end_ts)` предсказать `score` —
вероятность того, что кука принадлежит сервису автоматизированного сбора данных.
Метрика — `Precision @ Recall >= 0.70`, считается по кукам, код взят из приложенного `metric.py`.

Ноутбук выполняется сверху вниз и создаёт `submission.csv`. Полный прогон — около 45 минут,
основное время уходит на валидацию и проверку гипотез; финальная модель обучается за пару минут.

---

## Описание подхода

**Признаки.** Готовых нет, всё собирается из `events.csv.gz`. Двенадцать групп, 113 признаков.
99 описывают поведение куки внутри окна: робот дёшево наращивает объём и дорого имитирует
нерегулярность, поэтому вес отдан не счётчикам, а форме распределения интервалов, неравномерности
обхода каталога и пиковой интенсивности. Ещё 14 описывают **репутацию просмотренных объявлений**:
какая доля их прошлых посетителей оказалась ботами. Это единственная группа, использующая разметку
других куки, и она даёт самый большой прирост за всю задачу (раздел 11).

**Валидация.** Тест лежит позже трейна, поэтому валидация временна́я: пять расширяющихся
двухдневных окон для поведенческих признаков и три длинных фолда для репутации — у коротких фолдов
слишком мало ботов, чтобы различать эффекты в 0.03.

**Модель.** `HistGradientBoostingClassifier` из scikit-learn. Итоговый score — смесь нормированных
рангов: 0.7 на модель по всем данным и 0.3 на среднее 24 моделей на бутстрэп-выборках.

**Результат.** Предыдущая версия (99 признаков, медиана рангов) — 0.827 на валидации и **0.839** на
скрытом тесте. Текущая на скрытом тесте не проверялась: на тех же пяти окнах она даёт 0.849 против
0.827, на длинных фолдах — 0.854 против 0.805 у одиночной модели.

**Сторонние идеи.** Накопительный LLR по «очищенному» пулу объявлений (раздел 11.6) взят из
открытого решения [nefedovdima/avito-bot-detection](https://github.com/nefedovdima/avito-bot-detection);
реализация своя, его код в ноутбук не включён.

**Использованные библиотеки:** numpy, pandas, scikit-learn, scipy. Версии — в `requirements.txt`,
`random_state` зафиксирован, повторный прогон даёт побайтово идентичный файл.

## 1. Окружение

In [1]:
import sys, platform, hashlib, warnings
import numpy as np
import pandas as pd
import sklearn
from scipy.stats import rankdata
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

from metric import precision_at_recall

warnings.filterwarnings('ignore')
SEED = 42

print('Python      :', sys.version.split()[0], '|', platform.platform())
print('numpy       :', np.__version__)
print('pandas      :', pd.__version__)
print('scikit-learn:', sklearn.__version__)

Python      : 3.13.15 | Windows-10-10.0.19045-SP0
numpy       : 2.4.4
pandas      : 3.0.2
scikit-learn: 1.8.0


## 2. Загрузка и первый взгляд

In [2]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test  = pd.read_csv('data/test.csv',  parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
raw   = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

y = train.target.values
print('train', train.shape, '| test', test.shape, '| events', raw.shape)
print('доля ботов:', round(y.mean(), 4))
print('окна train:', train.window_start_ts.min().date(), '-', train.window_start_ts.max().date())
print('окна test :', test.window_start_ts.min().date(), '-', test.window_start_ts.max().date())

train (11091, 5) | test (4909, 4) | events (328905, 14)
доля ботов: 0.0811
окна train: 2026-04-06 - 2026-04-19
окна test : 2026-04-20 - 2026-04-26


Окна теста идут строго после окон трейна. Отсюда первое решение: валидация только временна́я.
Случайный сплит дал бы оптимистичную и бесполезную оценку, потому что модель увидела бы будущее.

In [3]:
print(raw.event_name.value_counts(), '\n')
print(raw.platform.value_counts(), '\n')
print('доля пропусков:'); print(raw.isna().mean().round(3), '\n')
print('полных дублей строк:', raw.duplicated().sum())
print('события отсортированы по времени внутри куки:',
      raw.groupby('cookie_id').event_ts.is_monotonic_increasing.all())

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

доля пропусков:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64 

полных дублей строк: 4863
события отсортированы по времени внутри куки: 

Три вещи, которые определяют шаг очистки.

**`platform` записана в разном регистре**: `WEB` / `Web` / `web`, `ANDROID` / `Android` / `android`,
`IOS` / `iOS` / `ios` / `iphone`. Без нормализации получится категориальная переменная с одиннадцатью
уровнями вместо четырёх.

**Пропуски структурные, а не случайные.** `search_query` и `search_page` пусты там, где событие не
является поиском, `pointer_x/y` — там, где курсора не существует (мобильные платформы). Это не
«данные потеряли», а «такого поля у этого события не бывает». Заполнять медианой нельзя: для
признака вида «разброс координат курсора» пропуск означает «курсора не было вовсе», и если забить
его нулём, это слипнется с «курсор был и не двигался» — а это разные вещи. Поэтому пропуски
оставляю как есть, `HistGradientBoostingClassifier` обрабатывает их нативно, отправляя в ту ветку,
которая лучше делит выборку.

**Дубли и порядок строк.** 4863 полностью одинаковые строки — артефакт выгрузки, удаляю: иначе
поедут все счётчики и интервалы между событиями. Файл не отсортирован, а без сортировки по
`(cookie_id, event_ts)` любой `diff()` по времени даст мусор.

In [4]:
events = raw.drop_duplicates().copy()
events['platform'] = (events.platform.astype('string').str.strip().str.lower()
                      .replace({'iphone': 'ios'}))
events = events.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)

print(len(raw), '->', len(events), f'(удалено {len(raw) - len(events)})')
print(sorted(events.platform.unique()))

328905 -> 324042 (удалено 4863)
['android', 'desktop', 'ios', 'web']


## 3. Окно наблюдения

Признаки считаются только по событиям внутри окна — требование условия, а не выбор. Левая граница
включается, правая нет.

In [5]:
def in_window(ev, meta):
    e = ev.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    e = e[(e.event_ts >= e.window_start_ts) & (e.event_ts < e.window_end_ts)]
    return e.sort_values(['cookie_id', 'event_ts'], kind='mergesort').reset_index(drop=True)

ev_tr = in_window(events, train)
ev_te = in_window(events, test)
all_ev = pd.concat([ev_tr, ev_te], ignore_index=True)

print('событий в окнах:', len(ev_tr), '/', len(ev_te))
print('куки без событий:', train.cookie_id.nunique() - ev_tr.cookie_id.nunique(),
      '/', test.cookie_id.nunique() - ev_te.cookie_id.nunique())

событий в окнах: 195484 / 88349
куки без событий: 0 / 0


## 4. Схема валидации

Пять расширяющихся окон: валидация на двух днях, обучение на всём, что строго раньше. Первый фолд
всегда слабый — обучение идёт на четырёх днях, — но это нехватка данных, а не дефект модели: у
финальной модели весь трейн.

Сначала я мерил на двух фолдах и это оказалось ошибкой. На двух фолдах суммарно около 260 ботов, и
разброс между соседними конфигурациями сопоставим с шумом, то есть решения принимались по случайным
колебаниям. Пять фолдов дают вдвое более устойчивую оценку.

In [6]:
FOLDS = [('2026-04-11', '2026-04-13'), ('2026-04-13', '2026-04-15'), ('2026-04-15', '2026-04-17'),
         ('2026-04-17', '2026-04-19'), ('2026-04-19', '2026-04-20')]

PARAMS = dict(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=20,
              l2_regularization=1.0, early_stopping=False, random_state=SEED)

WS = train.window_start_ts.values

def folds():
    for a, b in FOLDS:
        yield (WS >= np.datetime64(a)) & (WS < np.datetime64(b)), WS < np.datetime64(a)

def cv(X, cols, params=PARAMS, ret=False):
    """Средний P@R0.7 по временным фолдам для одной модели."""
    res = []
    for va, tr in folds():
        m = HistGradientBoostingClassifier(**params).fit(X.loc[tr, cols], y[tr])
        res.append(precision_at_recall(y[va], m.predict_proba(X.loc[va, cols])[:, 1]))
    return (round(float(np.mean(res)), 4), [round(r, 4) for r in res]) if ret else float(np.mean(res))

## 5. Baseline

Два счётчика и случайный лес, как в quickstart. Нужен только как точка отсчёта.

In [7]:
g = ev_tr.groupby('cookie_id')
base = (train[['cookie_id']]
        .merge(pd.DataFrame({'n_events': g.size(), 'u_item': g.item_id.nunique()}).reset_index(),
               on='cookie_id', how='left').fillna(0))

res = []
for va, tr in folds():
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=SEED)
    rf.fit(base.loc[tr, ['n_events', 'u_item']], y[tr])
    res.append(precision_at_recall(y[va], rf.predict_proba(base.loc[va, ['n_events', 'u_item']])[:, 1]))

print('baseline P@R0.7:', round(np.mean(res), 4), '| константа:', round(y.mean(), 4))

baseline P@R0.7: 0.0946 | константа: 0.0811


Baseline практически неотличим от константы. Объём активности сам по себе бота не выдаёт, и это
логично: человек, который полдня листает объявления, тоже генерирует много событий. Копать надо не в
«сколько», а в «как».

## 6. Признаки

Каждая группа — отдельная функция, принимает события внутри окна и возвращает таблицу с индексом
`cookie_id`. Так их удобно подключать по одной и мерить эффект.

In [8]:
EVENTS = ['item_view', 'search_results_view', 'photo_swipe', 'favorite_add', 'seller_page_view',
          'contact_phone_show', 'captcha_shown', 'login', 'contact_chat_open', 'contact_message_sent']

def entropy(counts):
    c = np.asarray(counts, float); c = c[c > 0]
    if c.sum() == 0:
        return 0.0
    p = c / c.sum()
    return float(-(p * np.log(p)).sum())

def counts_table(ev):
    return pd.crosstab(ev.cookie_id, ev.event_name).reindex(columns=EVENTS, fill_value=0)

### 6.1 Типы событий

Счётчики и доли. Доли важнее: они не зависят от общего объёма активности. Плюс энтропия по типам —
насколько равномерно размазана активность.

In [9]:
def f_counts(ev):
    ct = counts_table(ev)
    n = ev.groupby('cookie_id').size()
    f = pd.DataFrame(index=ct.index)
    for c in EVENTS:
        f['cnt_' + c] = ct[c]
        f['shr_' + c] = ct[c] / n
    f['n_events'] = n
    f['n_event_types'] = (ct > 0).sum(axis=1)
    f['event_type_entropy'] = ct.apply(lambda r: entropy(r.values), axis=1)
    return f

### 6.2 Воронка

Парсер много смотрит и почти не взаимодействует, человек наоборот: посмотрел пять объявлений, в два
написал. Считаю отношения к числу просмотров, а не абсолюты — они устойчивы к разному уровню
активности.

In [10]:
def f_ratios(ev):
    ct = counts_table(ev)
    v = ct.item_view.clip(lower=1)
    f = pd.DataFrame(index=ct.index)
    f['r_phone_per_view']  = ct.contact_phone_show / v
    f['r_swipe_per_view']  = ct.photo_swipe / v
    f['r_fav_per_view']    = ct.favorite_add / v
    f['r_seller_per_view'] = ct.seller_page_view / v
    f['r_contact_any']     = (ct.contact_chat_open + ct.contact_message_sent + ct.contact_phone_show) / v
    f['r_search_per_view'] = ct.search_results_view / v
    f['has_login']   = (ct.login > 0).astype(int)
    f['has_captcha'] = (ct.captcha_shown > 0).astype(int)
    return f

### 6.3 Разнообразие

Что именно смотрит кука. Парсер обходит каталог вширь: много категорий, много городов, мало
повторов. Человек ищет конкретную вещь и возвращается к одним и тем же объявлениям.

In [11]:
def f_diversity(ev):
    g = ev.groupby('cookie_id'); ct = counts_table(ev)
    f = pd.DataFrame(index=g.size().index)
    f['u_item']        = g.item_id.nunique()
    f['u_category']    = g.item_category.nunique()
    f['u_location']    = g.item_location.nunique()
    f['u_query']       = g.search_query.nunique()
    f['u_seller_type'] = g.seller_type.nunique()
    f['item_repeat_rate'] = 1 - f.u_item / ct.item_view.clip(lower=1)
    f['cat_per_item']     = f.u_category / f.u_item.clip(lower=1)   # "ширина" обхода
    f['loc_per_item']     = f.u_location / f.u_item.clip(lower=1)
    f['query_per_search'] = f.u_query / ct.search_results_view.clip(lower=1)
    return f

### 6.4 Пагинация

Человек редко уходит дальше второй страницы выдачи, парсер идёт по страницам подряд.
`page_seq_share` — доля переходов ровно на +1 внутри одного запроса.

In [12]:
def f_pages(ev):
    sp = ev.dropna(subset=['search_page'])
    f = pd.DataFrame(index=pd.Index(sorted(ev.cookie_id.unique()), name='cookie_id'))
    if len(sp) == 0:
        for c in ['page_max', 'page_mean', 'page_deep_share', 'page_seq_share']:
            f[c] = np.nan
        return f
    g = sp.groupby('cookie_id')
    f['page_max']  = g.search_page.max()
    f['page_mean'] = g.search_page.mean()
    f['page_deep_share'] = g.search_page.apply(lambda x: (x > 1).mean())
    sp2 = sp.sort_values(['cookie_id', 'search_query', 'event_ts'])
    step = sp2.groupby(['cookie_id', 'search_query']).search_page.diff()
    f['page_seq_share'] = (step == 1).groupby(sp2.cookie_id).mean()
    return f

### 6.5 Время

Главная содержательная группа. У скрипта с фиксированной паузой интервалы между событиями
кучкуются, у человека размазаны: то очередь событий за секунду, то пауза на полчаса.

Отдельно стоит берстность `(σ−μ)/(σ+μ)`. У пуассоновского потока, где события независимы, σ ≈ μ и
индекс около нуля; у строго регулярного σ → 0 и индекс → −1; у пачечного → +1. Одно безразмерное
число вместо трёх коррелирующих статистик, и сравнивать можно куки с разной активностью.

In [13]:
def f_time(ev):
    ev = ev.assign(_t=ev.event_ts.astype('int64') // 10**9)
    g = ev.groupby('cookie_id')
    dt = g._t.diff()
    dg = dt.groupby(ev.cookie_id)
    f = pd.DataFrame(index=g.size().index)
    f['dt_mean']   = dg.mean()
    f['dt_median'] = dg.median()
    f['dt_std']    = dg.std()
    f['dt_min']    = dg.min()
    f['dt_max']    = dg.max()
    f['dt_cv']     = f.dt_std / f.dt_mean.replace(0, np.nan)
    f['burstiness'] = (f.dt_std - f.dt_mean) / (f.dt_std + f.dt_mean).replace(0, np.nan)
    f['dt_fast_share'] = dg.apply(lambda x: (x < 1).mean())
    f['dt_lt3_share']  = dg.apply(lambda x: (x < 3).mean())
    # доля самого частого интервала с округлением до секунды — прямая мера "тикания по таймеру"
    f['dt_mode_share'] = dg.apply(lambda x: x.round().value_counts(normalize=True).max()
                                  if x.notna().any() else np.nan)
    f['span_sec'] = g._t.max() - g._t.min()
    f['rate_per_hour'] = g.size() / (f.span_sec / 3600).replace(0, np.nan)
    f['n_sessions'] = (dt.isna() | (dt > 1800)).astype(int).groupby(ev.cookie_id).sum()
    f['events_per_session'] = g.size() / f.n_sessions.clip(lower=1)
    hour = ev.event_ts.dt.hour
    f['hours_active'] = hour.groupby(ev.cookie_id).nunique()
    f['hour_entropy'] = hour.groupby(ev.cookie_id).apply(lambda x: entropy(x.value_counts().values))
    f['night_share']  = hour.groupby(ev.cookie_id).apply(lambda x: x.isin(range(6)).mean())
    return f

### 6.6 Платформа и user-agent

Сырой UA как категорию брать нельзя: в тесте полезут строки, которых не было в трейне, и модель про
них ничего не знает. Разбираю на устойчивые куски: какая ОС, какая мажорная версия, нет ли явных
маркеров автоматизации. Плюс две идеи — сколько разных UA у куки за сутки (ротация это аномалия) и
согласуется ли UA с заявленной платформой.

In [14]:
def parse_ua(s):
    txt = s.fillna('').astype(str); low = txt.str.lower()
    out = pd.DataFrame(index=s.index)
    out['len']      = txt.str.len()
    out['android']  = low.str.contains('android').astype(int)
    out['ios']      = low.str.contains('iphone|ipad|ios').astype(int)
    out['windows']  = low.str.contains('windows').astype(int)
    out['mac']      = low.str.contains('macintosh|mac os').astype(int)
    out['linux']    = (low.str.contains('linux') & ~low.str.contains('android')).astype(int)
    out['headless'] = low.str.contains('headless|phantom|electron').astype(int)
    out['lib']      = low.str.contains('curl|python|requests|go-http|okhttp|scrapy|java/|wget').astype(int)
    out['botword']  = low.str.contains('bot|spider|crawler').astype(int)
    out['major_ver'] = pd.to_numeric(
        txt.str.extract(r'(?:Chrome|Firefox|Version)/(\d+)', expand=False), errors='coerce')
    return out

def f_platform_ua(ev):
    g = ev.groupby('cookie_id'); n = g.size()
    f = pd.DataFrame(index=n.index)
    f['u_platform']   = g.platform.nunique()
    f['u_user_agent'] = g.user_agent.nunique()
    pf = pd.crosstab(ev.cookie_id, ev.platform)
    for p in ['desktop', 'web', 'android', 'ios']:
        f['plat_shr_' + p] = (pf[p] / n) if p in pf.columns else 0.0
    ua = parse_ua(ev.user_agent); ua['cookie_id'] = ev.cookie_id.values
    f = f.join(ua.groupby('cookie_id').mean(numeric_only=True).add_prefix('ua_'))
    f['ua_plat_mismatch'] = ((f.ua_android - f.plat_shr_android).abs()
                             + (f.ua_ios - f.plat_shr_ios).abs()) / 2
    return f

### 6.7 Курсор

Координаты есть только у части событий, поэтому отдельно считаю заполненность, отдельно статистики
по непустым. Эмулятор переиспользует одни и те же точки, живая рука так не умеет.

In [15]:
def f_pointer(ev):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame(index=g.size().index)
    f['pointer_fill'] = g.pointer_x.apply(lambda x: x.notna().mean())
    px = ev.dropna(subset=['pointer_x', 'pointer_y'])
    if len(px):
        gp = px.groupby('cookie_id')
        f['ptr_x_std'] = gp.pointer_x.std()
        f['ptr_y_std'] = gp.pointer_y.std()
        f['ptr_unique_share'] = gp.apply(lambda d: len(set(zip(d.pointer_x, d.pointer_y))) / len(d),
                                         include_groups=False)
    return f

### 6.8 Метаданные куки

Единственное, что есть помимо событий, — когда кука создана. Свежая кука с большой активностью
подозрительна.

День недели и час начала окна я пробовал и **выбросил**: они описывают нарезку данных, а не
поведение куки. Модель по ним частично запоминает, из какого дня пример, а в тесте дни другие.
Проверка в разделе 9.

In [16]:
def f_meta(meta):
    m = meta.set_index('cookie_id')
    age = (m.window_start_ts - m.cookie_created_at).dt.total_seconds() / 86400
    return pd.DataFrame({'cookie_age_days': age, 'cookie_fresh': (age < 1).astype(int)}, index=m.index)

### 6.9 Концентрация обхода

`u_category` считает, сколько разных категорий смотрела кука. Но важнее не количество, а
**равномерность**: у парсера обход ровный, у человека один явный фокус и хвост случайных заходов.
Индекс Херфиндаля `Σp²` это ловит: 1 — всё в одной категории, ближе к нулю — размазано.

In [17]:
def hhi(x):
    v = x.dropna()
    if len(v) == 0:
        return np.nan
    p = v.value_counts(normalize=True).values
    return float((p ** 2).sum())

def f_conc(ev):
    g = ev.groupby('cookie_id')
    return pd.DataFrame({
        'hhi_category':  g.item_category.apply(hhi),
        'hhi_location':  g.item_location.apply(hhi),
        'hhi_query':     g.search_query.apply(hhi),
        'items_per_cat': g.item_id.nunique() / g.item_category.nunique().clip(lower=1),
    })

### 6.10 Пиковая интенсивность

Средний темп у меня уже есть (`rate_per_hour`), но показателен не средний темп, а **пик**: всплеск в
десятки событий за минуту человеку физически недоступен, а скрипту достаётся даром. Отношение пика к
медиане нормирует это на общий уровень активности.

In [18]:
def f_peak(ev):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame(index=g.size().index)
    per_min = ev.assign(_m=ev.event_ts.dt.floor('1min')).groupby(['cookie_id', '_m']).size().groupby('cookie_id')
    f['max_ev_per_min'] = per_min.max()
    f['med_ev_per_min'] = per_min.median()
    f['peak_to_med_1m'] = f.max_ev_per_min / f.med_ev_per_min.replace(0, np.nan)
    per_5m = ev.assign(_m=ev.event_ts.dt.floor('5min')).groupby(['cookie_id', '_m']).size().groupby('cookie_id')
    f['max_ev_per_5min'] = per_5m.max()
    f['peak_to_med_5m']  = per_5m.max() / per_5m.median().replace(0, np.nan)
    f['active_minutes']  = ev.assign(_m=ev.event_ts.dt.floor('1min')).groupby('cookie_id')._m.nunique()
    return f

### 6.11 Контекст по корпусу

Все признаки выше описывают куку изолированно. Но есть и контекст: насколько популярны объявления,
которые она смотрит, и сколько других куки сидят на том же user-agent. Логика: парсер обходит
массово и попадает на те же позиции, что и другие парсеры, человек смотрит свою частную выборку.

Меток здесь нет, только события, поэтому утечки таргета не возникает. С временем сложнее — разбираю
это отдельно в разделе 8, там главный сюжет.

In [19]:
def ua_norm(s):
    """Схлопываем версии в 'N': Chrome/119.0.0.0 и Chrome/120.0.0.0 — один агент."""
    return s.fillna('').astype(str).str.lower().str.replace(r'\d+(\.\d+)*', 'N', regex=True)

def f_cross(ev_src, idx):
    """Контекст по корпусу. ev_src задаёт, по каким событиям он считается, — ключевой параметр."""
    e = ev_src.assign(_ua=ua_norm(ev_src.user_agent))
    ck_ua = e.groupby('cookie_id')._ua.agg(lambda s: s.mode().iat[0] if len(s) else '')
    out = pd.DataFrame(index=idx)
    out['ua_share_count'] = ck_ua.map(ck_ua.value_counts()).reindex(idx)

    iv = e[e.event_name == 'item_view']
    cip = iv.assign(p=iv.item_id.map(iv.groupby('item_id').cookie_id.nunique())).groupby('cookie_id').p
    out['item_pop_mean'] = cip.mean().reindex(idx)
    out['item_pop_max']  = cip.max().reindex(idx)

    sq = e.dropna(subset=['search_query'])
    cqp = sq.assign(p=sq.search_query.map(sq.groupby('search_query').cookie_id.nunique())).groupby('cookie_id').p
    out['query_pop_mean'] = cqp.mean().reindex(idx)

    lv = e.dropna(subset=['item_location'])
    clp = lv.assign(p=lv.item_location.map(lv.groupby('item_location').cookie_id.nunique())).groupby('cookie_id').p
    out['loc_pop_mean'] = clp.mean().reindex(idx)
    return out

### Сборка

In [20]:
%%time
def build(ev, meta):
    parts = [f_counts(ev), f_ratios(ev), f_diversity(ev), f_pages(ev),
             f_time(ev), f_platform_ua(ev), f_pointer(ev)]
    X = pd.concat(parts, axis=1).join(f_meta(meta), how='right')
    X['n_events'] = X.n_events.fillna(0)          # куки без событий: активности честно ноль
    return X.reindex(meta.set_index('cookie_id').index)

X_tr = build(ev_tr, train)
X_te = build(ev_te, test)
CORE = list(X_tr.columns)

for name, ftr, fte in [('conc', f_conc(ev_tr), f_conc(ev_te)),
                       ('peak', f_peak(ev_tr), f_peak(ev_te)),
                       ('cross', f_cross(all_ev, X_tr.index).add_prefix('cr_'),
                                 f_cross(all_ev, X_te.index).add_prefix('cr_'))]:
    X_tr = X_tr.join(ftr.reindex(X_tr.index))
    X_te = X_te.join(fte.reindex(X_te.index))
    globals()[name.upper()] = list(ftr.columns)

X_tr, X_te = X_tr.reset_index(), X_te.reset_index()
FEATURES = CORE + CONC + PEAK + CROSS

assert list(X_tr.cookie_id) == list(train.cookie_id)
assert list(X_te.cookie_id) == list(test.cookie_id)
print(len(CORE), 'базовых +', len(CONC), 'концентрация +', len(PEAK), 'пик +',
      len(CROSS), 'контекст =', len(FEATURES))

84 базовых + 4 концентрация + 6 пик + 5 контекст = 99
CPU times: total: 54.1 s
Wall time: 54.8 s


## 7. Замер групп по очереди

Добавляю группу, смотрю на среднее по пяти фолдам.

In [21]:
%%time
GROUPS = [('счётчики', list(f_counts(ev_tr).columns)),
          ('воронка', list(f_ratios(ev_tr).columns)),
          ('разнообразие', list(f_diversity(ev_tr).columns)),
          ('пагинация', list(f_pages(ev_tr).columns)),
          ('время', list(f_time(ev_tr).columns)),
          ('платформа+UA', list(f_platform_ua(ev_tr).columns)),
          ('курсор', list(f_pointer(ev_tr).columns)),
          ('мета куки', ['cookie_age_days', 'cookie_fresh']),
          ('концентрация', CONC),
          ('пик', PEAK),
          ('контекст', CROSS)]

cols = []
for name, gcols in GROUPS:
    cols = [c for c in FEATURES if c in set(cols) | set(gcols)]
    print(f'{name:14s} {len(cols):3d} признаков -> {cv(X_tr, cols):.4f}')

счётчики        23 признаков -> 0.1537
воронка         31 признаков -> 0.1562
разнообразие    40 признаков -> 0.2137
пагинация       44 признаков -> 0.2627
время           61 признаков -> 0.3180
платформа+UA    78 признаков -> 0.3409
курсор          82 признаков -> 0.4920
мета куки       84 признаков -> 0.5399
концентрация    88 признаков -> 0.5835
пик             94 признаков -> 0.6501
контекст        99 признаков -> 0.7597
CPU times: total: 10min 25s
Wall time: 2min 54s


Три наблюдения.

**Разнообразие даёт первый настоящий скачок.** То, *что* кука смотрит, информативнее того,
*сколько*.

**Курсор даёт самый большой прирост из «внутренних» групп** — это стоит проверить отдельно, слишком
красиво.

**Контекст по корпусу даёт ещё больше** — и это стоит проверить особенно тщательно.

In [22]:
%%time
ptr = list(f_pointer(ev_tr).columns)
print('только курсор  :', round(cv(X_tr, ptr), 4))
print('всё без курсора:', round(cv(X_tr, [c for c in FEATURES if c not in ptr]), 4))
print('всё            :', round(cv(X_tr, FEATURES), 4))

только курсор  : 0.1125
всё без курсора: 0.6695
всё            : 0.7597
CPU times: total: 2min 49s
Wall time: 44.8 s


Сам по себе курсор почти ничего не даёт, но без него полный набор заметно падает. Значит это не
признак-читер, а **взаимодействие**: разброс координат разделяет куки только в сочетании с профилем
активности. Отсюда практический вывод: permutation importance на этих данных врёт — переставляешь
один признак, а информацию несут соседи, и важность выходит заниженной. Считать надо группами, что я
и делаю.

## 8. Проверка контекста на утечку во времени

Подозрение. `item_pop_mean` для куки из 7 апреля считается по всем событиям корпуса, включая конец
апреля. То есть признак смотрит в будущее относительно окна наблюдения этой куки, а условие требует,
чтобы признаки были доступны на момент `window_end_ts`.

Проверяю в лоб: пересчитываю контекст для каждого фолда, используя только события, случившиеся **до
начала валидационного окна**.

In [23]:
%%time
inner = CORE + CONC + PEAK
rows = []
for mode in ['без контекста', 'контекст по всему корпусу', 'контекст только из прошлого']:
    res = []
    for (va, tr), (a, b) in zip(folds(), FOLDS):
        if mode == 'без контекста':
            Xf, cols = X_tr, inner
        else:
            src = all_ev if mode == 'контекст по всему корпусу' else all_ev[all_ev.event_ts < np.datetime64(a)]
            cr = f_cross(src, pd.Index(train.cookie_id)).add_prefix('cr_').reset_index(drop=True)
            Xf = pd.concat([X_tr[inner], cr], axis=1); cols = inner + CROSS
        m = HistGradientBoostingClassifier(**PARAMS).fit(Xf.loc[tr, cols], y[tr])
        res.append(precision_at_recall(y[va], m.predict_proba(Xf.loc[va, cols])[:, 1]))
    rows.append({'вариант': mode, 'P@R0.7': round(np.mean(res), 4), 'по фолдам': [round(x, 4) for x in res]})

pd.DataFrame(rows)

CPU times: total: 4min 10s
Wall time: 1min 29s


,вариант,P@R0.7,по фолдам
0,без контекста,0.6501,"[0.5337, 0.6127, 0.5871, 0.7477, 0.7692]"
1,контекст по всему корпусу,0.7597,"[0.5598, 0.8529, 0.8318, 0.7434, 0.8108]"
2,контекст только из прошлого,0.6081,"[0.4703, 0.6286, 0.6154, 0.6288, 0.6977]"


Числа: без контекста 0.650, по всему корпусу 0.760, только из прошлого 0.608.

Первое прочтение — «весь прирост держится на заглядывании вперёд, признак надо выбросить». Я так и
решил, и **это было ошибкой**, которую показала отправка на лидерборд: вариант с контекстом дал там
0.835 против 0.726 без него, то есть прирост перенёсся полностью.

Где ошибка в рассуждении. Ограничивая контекст прошлым, я убирал не только заглядывание вперёд, но и
**объём данных, по которым оценивается популярность**. На ранних фолдах это считанные дни корпуса,
оценки становятся шумными, и признак деградирует по совершенно другой причине.

Содержательно правильная рамка: популярность объявления — свойство корпуса, а не куки. События всех
тестовых куки нам выданы, значит для теста популярность оценивается по тому же полному пулу, что и
для трейна. Признак симметричен между обучением и применением, меток не использует, а валидационный
фолд с полным контекстом воспроизводит ситуацию теста точнее, чем причинный вариант. Формально
ограничение про `window_end_ts` он натягивает — это честное ограничение решения, — но по существу
работает.

Урок общий: причинная проверка признака должна менять **только** причинность, а не объём данных
заодно. Иначе результат неинтерпретируем.

## 9. Что проверено и отброшено

Из примерно семидесяти проверенных гипотез рабочими оказались пять. Здесь — самое поучительное из
отброшенного, с числами.

In [24]:
%%time
# Target encoding user-agent (fold-safe: кодировка считается только по train-части фолда)
def te_fit(keys, yy, prior, sm):
    d = pd.DataFrame({'k': keys, 'y': yy}).groupby('k').y.agg(['sum', 'count'])
    return (d['sum'] + prior * sm) / (d['count'] + sm)

ua_key = ua_norm(all_ev.user_agent).groupby(all_ev.cookie_id).agg(lambda s: s.mode().iat[0])
keys = ua_key.reindex(train.cookie_id).fillna('').values
print('различных нормализованных UA во всём корпусе:', ua_key.nunique())

res = []
for va, tr in folds():
    pri = y[tr].mean()
    enc = te_fit(keys[tr], y[tr], pri, 30)
    Xf = X_tr[FEATURES].copy()
    Xf['te_ua'] = pd.Series(keys).map(enc).fillna(pri).values
    m = HistGradientBoostingClassifier(**PARAMS).fit(Xf.loc[tr], y[tr])
    res.append(precision_at_recall(y[va], m.predict_proba(Xf.loc[va])[:, 1]))
print('+ TE user_agent:', round(np.mean(res), 4))

различных нормализованных UA во всём корпусе: 19
+ TE user_agent: 0.7677
CPU times: total: 1min 12s
Wall time: 20.1 s


Одиночная модель показывает крошечный плюс, но это ровно тот масштаб, который на пяти фолдах
неотличим от шума, а структурный аргумент против — в числе различных агентов: после схлопывания
версий их во всём корпусе меньше двух десятков. Это не высококардинальная категория, ради которой
target encoding и придумывают, а обычный фактор на 19 уровней, который модель и так достаёт из
имеющихся `ua_*` признаков. Признак, способный лишь переформулировать уже имеющуюся информацию,
добавляет дисперсии, не добавляя сигнала. Та же картина с target encoding дня недели.

Календарные признаки (`window_dow`, `window_hour_start`) я проверяю ниже, в разделе 10, вместе с
бэггингом — там видно, почему одиночная модель по таким вопросам ненадёжный судья.

In [25]:
%%time
# Отбор признаков по важности (важности считаются только на train-части фолда)
for k in [30, 45]:
    res = []
    for va, tr in folds():
        rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=SEED)
        rf.fit(X_tr.loc[tr, FEATURES].fillna(-999), y[tr])
        top = list(pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False).head(k).index)
        m = HistGradientBoostingClassifier(**PARAMS).fit(X_tr.loc[tr, top], y[tr])
        res.append(precision_at_recall(y[va], m.predict_proba(X_tr.loc[va, top])[:, 1]))
    print(f'топ-{k}: {np.mean(res):.4f}')
print(f'все {len(FEATURES)}: {cv(X_tr, FEATURES):.4f}')

топ-30: 0.7584
топ-45: 0.7426
все 99: 0.7597
CPU times: total: 4min 10s
Wall time: 58.2 s


Обрезать набор нельзя: чем меньше признаков, тем хуже. Слабые поодиночке работают в связке — ровно то
же, что показала проверка курсора.

**Вырожденный признак.** Стоит посмотреть на `n_sessions`: у всех куки он равен единице. Разрыв
больше 30 минут внутри суточного окна не встречается ни разу, то есть `n_sessions` — константа, а
`events_per_session` просто дублирует `n_events`. Целая идея с сессиями на этих данных не работает,
хотя выглядела разумной.

In [26]:
print(X_tr.n_sessions.value_counts())

n_sessions
1    11091
Name: count, dtype: int64


**Что ещё проверено тем же способом и не дало прироста** (все в пределах ±0.02 от базы или хуже):

* квантили и скошенность интервалов, автокорреляция интервалов на лаге 1, энтропия округлённых
  интервалов, тренд темпа по ходу окна;
* временна́я концентрация: HHI по минутным и часовым корзинам, доля событий в самой загруженной
  минуте и часе, максимум за 10 секунд, длина и число пачек событий с интервалом до 2 секунд,
  коэффициент Джини по интервалам — всё семейство легло в 0.647-0.658 против 0.650, потому что
  полезный сигнал уже вытащен признаком `peak_to_med_1m`;
* энтропия переходов между типами событий (0.604), доля просмотров без предшествующего поиска
  (0.608), задержка от показа выдачи до перехода в объявление;
* динамика курсора: скорость, ускорение, извилистость траектории;
* текстовые свойства запросов: длина, число слов, доля с цифрами, переиспользование;
* новизна и синхронность объявлений (возраст объявления на момент просмотра, доля первых
  просмотров, число куки на том же объявлении в тот же час);
* сырой user_agent категорией, логарифмы объёмных признаков (деревья инвариантны к монотонным
  преобразованиям — число не меняется вообще), число дублей строк у куки, статистики длин сессий;
* перебор гиперпараметров: семь конфигураций, разброс ±0.05 — чистый шум.

Характерная ловушка: несколько из этих групп по отдельности как будто давали +0.02...+0.04, а все
вместе оказывались хуже исходного набора. Это диагностический признак того, что приросты лежали в
шуме. Под бэггингом, который дисперсию режет, они исчезли окончательно.

## 10. Ансамбль: бэггинг и свёртка рангов

Метрика ранжирующая и шумная, значит проблема в дисперсии, а не в смещении, — и бить надо именно по
ней. Обучаю модели на бутстрэп-выборках и усредняю **ранги**, а не вероятности: шкалы моделей
несопоставимы, а для ранжирующей метрики важен только порядок.

Дальше вопрос, чем свернуть ранги. Сравнение здесь **парное**: модели те же, данные те же,
различается только способ свёртки, поэтому вся дисперсия от выбора фолда и обучения сокращается, и
различия в 0.01 уже осмысленны.

In [27]:
%%time
def bagged(cols, va, tr, n_bag=8, seed=SEED):
    """Возвращает матрицу вероятностей (n_bag x n_val) для одного фолда."""
    Xa, ya, Xv = X_tr.loc[tr, cols], y[tr], X_tr.loc[va, cols]
    rng = np.random.default_rng(seed)
    out = []
    for i in range(n_bag):
        idx = rng.choice(len(Xa), len(Xa), replace=True)
        m = HistGradientBoostingClassifier(**{**PARAMS, 'random_state': 100 + i})
        out.append(m.fit(Xa.iloc[idx], ya[idx]).predict_proba(Xv)[:, 1])
    return np.array(out)

# Календарь проверяю здесь же: одиночная модель по таким вопросам ненадёжна.
CAL = ['window_dow', 'window_hour_start']
X_tr[CAL[0]] = train.window_start_ts.dt.dayofweek.values
X_tr[CAL[1]] = train.window_start_ts.dt.hour.values

rows = []
for name, cols in [('финальный набор', FEATURES), ('+ календарь', FEATURES + CAL)]:
    single, mean_r, med_r = [], [], []
    for (va, tr) in folds():
        P = bagged(cols, va, tr)
        R = np.array([rankdata(p) for p in P])
        mean_r.append(precision_at_recall(y[va], R.mean(0)))
        med_r.append(precision_at_recall(y[va], np.median(R, 0)))
        m = HistGradientBoostingClassifier(**PARAMS).fit(X_tr.loc[tr, cols], y[tr])
        single.append(precision_at_recall(y[va], m.predict_proba(X_tr.loc[va, cols])[:, 1]))
    rows.append({'набор': name, 'одна модель': round(np.mean(single), 4),
                 'бэггинг, среднее рангов': round(np.mean(mean_r), 4),
                 'бэггинг, медиана рангов': round(np.mean(med_r), 4)})

X_tr = X_tr.drop(columns=CAL)
pd.DataFrame(rows)

CPU times: total: 22min 4s
Wall time: 5min 40s


,набор,одна модель,"бэггинг, среднее рангов","бэггинг, медиана рангов"
0,финальный набор,0.7597,0.8089,0.8269
1,+ календарь,0.7791,0.8146,0.8141


Два вывода из одной таблицы.

**Бэггинг даёт заметный прирост, а медиана рангов — ещё около 0.02 сверху.** Медиана выигрывает у
среднего на всех пяти фолдах без исключения.

Механизм понятен. Одна модель из бутстрэпа может засунуть конкретную куку в резко нетипичное место
рейтинга — просто потому, что в её выборке не оказалось похожих примеров. Среднее тянется за таким
выбросом пропорционально 1/n, медиана его игнорирует. Это та же причина, по которой медиана
устойчивее среднего к тяжёлым хвостам.

**Календарь: одиночная модель ошибается.** С календарными признаками одна модель выглядит лучше, а
под бэггингом с медианой — хуже. Это ровно тот случай, ради которого стоит снижать дисперсию перед
тем, как принимать решение: кажущийся прирост в 0.02 у одиночной модели был шумом, и я едва не
оставил признак, описывающий нарезку данных, а не поведение куки. Календарь выброшен.

Общее правило, которое я вывел из этой задачи: если разница между вариантами меньше 0.03, одиночная
модель на пяти фолдах её не различает. Либо снижай дисперсию бэггингом, либо делай сравнение парным
(одни и те же модели, разная свёртка), либо не делай вывод.

## 11. Репутация объявлений по размеченной истории

До сих пор ни один признак не использовал разметку *других* куки. Группа `cr_*` считает, насколько
популярно объявление, но не кто именно его смотрел. Между тем задача про парсеров: сервис обходит
не случайные объявления, а ограниченный каталог — свою категорию, свой регион, свой прайс-лист. Если
объявление вчера смотрели известные боты, то кука, зашедшая на него сегодня, подозрительна сама по
себе, независимо от того, как она себя ведёт внутри окна.

Формально это target encoding по `item_id`, но с двумя особенностями, которые делают его нетривиальным.

**Ключ не у строки, а у множества.** У куки не одно объявление, а десятки, и репутацию каждого надо
свернуть в несколько чисел. Беру тринадцать агрегатов: сглаженную долю ботов среди посетителей,
её максимум и верхний квартиль, долю «горячих» объявлений (видел хоть один бот), долю «чистых»
(видели *только* боты), долю «холодных» (видели только люди), покрытие, логарифм отношения
правдоподобий, ставку со взвешиванием по редкости объявления и среднюю популярность.

**Утечка здесь не одна, а две.** Очевидная — своя метка не должна попасть в свой признак. Менее
очевидная и более опасная — несовпадение *зрелости* пула между обучением и инференсом; ей посвящён
отдельный замер ниже.

Сглаживание: $\mathrm{rate} = \dfrac{b + \alpha p_0}{n + \alpha}$, где $n$ — размеченных куки на
объявлении, $b$ — ботов среди них, $p_0$ — базовая доля ботов, $\alpha = 5$. Объявление с одним
посетителем-ботом не получает ставку 1.0, а подтягивается к базовой доле: при $n=b=1$ выходит
$1.4/6 \approx 0.23$, при $n=b=10$ уже $0.70$. Логарифм отношения правдоподобий
$\mathrm{LLR} = \log\frac{b+0.5}{B} - \log\frac{n-b+0.5}{H}$ (где $B$, $H$ — суммарные посещения
ботами и людьми) добавляет к этому вес редкости: объявление, которое видели двое и оба боты,
информативнее того, которое видели пятьсот человек и сорок ботов.

In [28]:
REP_STATS = ['rate_mean', 'rate_max', 'rate_q75', 'hot_share', 'hot_cnt', 'pure_share',
             'strong_share', 'cold_share', 'cov', 'llr_mean', 'llr_max', 'idf_rate', 'pop_mean']
REP = [f'ri_{s}' for s in REP_STATS]
P0_RATE, ALPHA = float(y.mean()), 5.0


def item_pairs(ev, meta):
    """Уникальные пары (кука, объявление) с днём окна этой куки."""
    p = ev.dropna(subset=['item_id'])[['cookie_id', 'item_id']].drop_duplicates()
    day = meta.set_index('cookie_id').window_start_ts.dt.normalize()
    return p.assign(day=p.cookie_id.map(day)).dropna(subset=['day'])


def day_counts(pairs_lab, meta_lab, y_lab):
    """(объявление, день) -> сколько размеченных куки его видели и сколько из них боты."""
    ys = pd.Series(np.asarray(y_lab), index=meta_lab.cookie_id)
    p = pairs_lab.assign(is_bot=pairs_lab.cookie_id.map(ys).astype(float))
    return (p.groupby(['item_id', 'day'], observed=True)
             .agg(n=('is_bot', 'size'), b=('is_bot', 'sum')).reset_index())


def rep_agg(p):
    """Пары с пуловыми счётчиками (n, b) -> тринадцать чисел на куку."""
    n, b = p.n.values.astype(float), p.b.values.astype(float)
    seen = n > 0
    rate = (b + ALPHA * P0_RATE) / (n + ALPHA)
    B, H = max(b.sum(), 1.0), max((n - b).sum(), 1.0)
    p = p.assign(
        rate=np.where(seen, rate, np.nan),
        hot=np.where(seen & (b > 0), 1.0, 0.0),
        pure=np.where(seen & (b > 0) & (b == n), 1.0, 0.0),   # видели только боты
        strong=np.where(seen & (b >= 2), 1.0, 0.0),
        cold=np.where(seen & (b == 0), 1.0, 0.0),             # видели только люди
        seen=seen.astype(float),
        w=np.where(seen, 1.0 / np.log(2.0 + n), 0.0),         # вес редкости
        wr=np.where(seen, rate / np.log(2.0 + n), 0.0),
        llr=np.where(seen, np.log((b + .5) / B) - np.log((n - b + .5) / H), np.nan),
        pop=n)
    g = p.groupby('cookie_id', observed=True)
    out = pd.DataFrame({
        'ri_rate_mean': g.rate.mean(), 'ri_rate_max': g.rate.max(),
        'ri_rate_q75': g.rate.quantile(.75), 'ri_hot_share': g.hot.mean(),
        'ri_hot_cnt': g.hot.sum(), 'ri_pure_share': g.pure.mean(),
        'ri_strong_share': g.strong.mean(), 'ri_cold_share': g.cold.mean(),
        'ri_cov': g.seen.mean(), 'ri_llr_mean': g.llr.mean(), 'ri_llr_max': g.llr.max(),
        'ri_idf_rate': g.wr.sum() / g.w.sum().replace(0.0, np.nan), 'ri_pop_mean': g.pop.mean()})
    return out


def _pool_before(dc, horizon):
    """Счётчики по объявлениям, накопленные строго раньше horizon."""
    return dc[dc.day < pd.Timestamp(horizon)].groupby('item_id', observed=True)[['n', 'b']].sum()


def _attach(pairs, pool):
    p = pairs.merge(pool, left_on='item_id', right_index=True, how='left')
    return p.fillna({'n': 0.0, 'b': 0.0})


def rep_asof(pairs, dc, horizon, index):
    """Пул = размеченные дни строго раньше horizon, один на все строки. Так считается на тесте."""
    return rep_agg(_attach(pairs, _pool_before(dc, horizon))).reindex(index).reset_index(drop=True)


def rep_lodo(pairs, dc, index):
    """Пул = все размеченные дни, кроме дня самой строки. Так считается на обучении."""
    tot = dc.groupby('item_id', observed=True)[['n', 'b']].sum()
    p = _attach(pairs, tot)
    p = p.merge(dc.rename(columns={'n': 'n_own', 'b': 'b_own'}), on=['item_id', 'day'], how='left')
    p[['n_own', 'b_own']] = p[['n_own', 'b_own']].fillna(0.0)
    p['n'] -= p.n_own
    p['b'] -= p.b_own
    return rep_agg(p).reindex(index).reset_index(drop=True)


def rep_past(pairs, dc, index):
    """Пул = дни строго раньше дня строки. Осторожный вариант, см. сравнение ниже."""
    parts = [rep_agg(_attach(pairs[pairs.day == d], _pool_before(dc, d)))
             for d in sorted(pairs.day.unique())]
    return pd.concat(parts).reindex(index).reset_index(drop=True)



# --- Накопительный LLR по «очищенному» пулу (идея из решения nefedovdima, раздел 11.6) ---
CP_TAU, CP_MIN_ITEMS, CP_WINDOW = 0.3, 3, 7


def cp_fit(pairs, hist_ids, labels):
    """Очищенный пул по истории и вероятности попасть в него для ботов и для людей."""
    h = pairs[pairs.cookie_id.isin(hist_ids)]
    yb = labels.reindex(h.cookie_id).to_numpy()
    bots, hum = h[yb == 1], h[yb == 0]
    nb = bots.groupby('item_id', observed=True).size()
    shared = bots.item_id.map(nb).to_numpy() > 1             # объявление видел ещё хоть один бот
    share = pd.Series(shared, index=bots.cookie_id.to_numpy()).groupby(level=0).mean()
    size = bots.groupby('cookie_id', observed=True).size().reindex(share.index)
    keep = share.index[(share >= CP_TAU) & (size >= CP_MIN_ITEMS)]   # «пуловые» боты
    pb = bots[bots.cookie_id.isin(keep)]
    pool = pb.groupby('item_id', observed=True).size()
    p_b = float((pb.item_id.map(pool).to_numpy() > 1).mean())    # попадание бота, не считая себя
    p_h = float(hum.item_id.isin(pool.index).mean())
    return pool, p_b, p_h


def cp_llr(pairs, cur_ids, pool, p_b, p_h, eps=1e-6):
    """m попаданий из k объявлений: m·log(p_b/p_h) + (k − m)·log((1 − p_b)/(1 − p_h))."""
    c = pairs[pairs.cookie_id.isin(cur_ids)]
    hit = pd.Series(c.item_id.isin(pool.index).to_numpy(), index=c.cookie_id.to_numpy()).groupby(level=0)
    m, k = hit.sum(), hit.size()
    a, b = np.clip(p_b, eps, 1 - eps), np.clip(p_h, eps, 1 - eps)
    return (m * np.log(a / b) + (k - m) * np.log((1 - a) / (1 - b))).reindex(cur_ids)


def cp_history(meta, ids, cutoff, window=CP_WINDOW):
    """Размеченные окна, закончившиеся к cutoff и начавшиеся не раньше чем за window дней до него."""
    h = meta.set_index('cookie_id').loc[ids]
    cutoff = pd.Timestamp(cutoff)
    return h.index[(h.window_end_ts <= cutoff) & (h.window_start_ts >= cutoff - pd.Timedelta(days=window))]


def cp_train(pairs, meta, labels):
    """Обучающие строки: история строго до дня строки. Почему не lodo — раздел 11.6."""
    ids = pd.Index(meta.cookie_id)
    day = meta.set_index('cookie_id').window_start_ts.dt.normalize()
    out = []
    for d, cur in day.groupby(day).groups.items():
        hist = cp_history(meta, ids, d)
        out.append(cp_llr(pairs, pd.Index(cur), *cp_fit(pairs, hist, labels)) if len(hist)
                   else pd.Series(np.nan, index=cur))
    return pd.concat(out).reindex(ids).to_numpy()


def cp_infer(pairs, cur_meta, lab_meta, labels, horizon):
    """Валидация и тест: одна история на весь блок, замороженная на horizon."""
    hist = cp_history(lab_meta, pd.Index(lab_meta.cookie_id), horizon)
    return cp_llr(pairs, pd.Index(cur_meta.cookie_id), *cp_fit(pairs, hist, labels)).to_numpy()


### 11.1 Живёт ли сигнал во времени

Прежде чем строить признак, надо проверить его срок годности. Тест отстоит от конца трейна на 1–7
дней: если репутация объявления протухает за сутки, для 26 апреля она будет бесполезна, и тогда
признак надо строить иначе — с затуханием, с поправкой на разрыв.

Замер прямой. Беру пул из дней 6–12 апреля, считаю единственный признак `hot_share` для кук
13–19 апреля и смотрю, как его AUC зависит от того, на сколько дней кука отстоит от конца пула.

In [29]:
from sklearn.metrics import roc_auc_score

CUT0 = pd.Timestamp('2026-04-13')
_pairs = item_pairs(ev_tr, train)
_dc = day_counts(_pairs[_pairs.day < CUT0], train, y)
_cur = _pairs[_pairs.day >= CUT0]
_rep = rep_asof(_cur, _dc, CUT0, _cur.cookie_id.unique())
_rep.index = _cur.cookie_id.unique()
_rep['день'] = train.set_index('cookie_id').window_start_ts.dt.normalize().reindex(_rep.index)
_rep['y'] = pd.Series(y, index=train.cookie_id).reindex(_rep.index)

rows = []
for d, s in _rep.groupby('день'):
    rows.append({'лаг, дней': (d - CUT0).days, 'куки': len(s), 'ботов': int(s.y.sum()),
                 'AUC одного признака': round(roc_auc_score(s.y, s.ri_hot_share), 3),
                 'hot_share у ботов': round(s.ri_hot_share[s.y == 1].mean(), 3),
                 'hot_share у людей': round(s.ri_hot_share[s.y == 0].mean(), 3)})
pd.DataFrame(rows)

,"лаг, дней",куки,ботов,AUC одного признака,hot_share у ботов,hot_share у людей
0,0,831,53,0.777,0.458,0.087
1,1,817,70,0.829,0.537,0.088
2,2,789,65,0.778,0.447,0.084
3,3,723,60,0.827,0.509,0.080
4,4,701,66,0.767,0.458,0.084
5,5,602,52,0.770,0.472,0.096
6,6,628,42,0.801,0.444,0.081


Затухания нет: AUC держится в коридоре 0.77–0.83 без тренда, разделение классов (примерно 0.47
против 0.09) одинаково на нулевом и на шестидневном лаге. Содержательно это значит, что парсеры
возвращаются к одному и тому же каталогу изо дня в день — иначе доля совпадений падала бы.

Два следствия. Первое: усложнять конструкцию под разрыв не нужно, для 26 апреля признак работает
так же, как для 20-го. Второе, менее очевидное, понадобится прямо сейчас: раз порядок дней внутри
пула не важен, пул можно собирать не только из прошлого.

### 11.2 Чем пул на обучении отличается от пула на инференсе

Здесь сидит ошибка, которая стоит дороже самого признака.

На тесте пул известен целиком: все четырнадцать размеченных дней лежат в прошлом относительно
любого тестового окна. А для обучающей строки «прошлое» — это только дни до неё: у куки 6 апреля
пула нет вовсе, у куки 7 апреля он в один день. Если строить признак буквально «только прошлое»,
модель учится на полупустом признаке, а на инференсе получает насыщенный. Покрытие (доля
объявлений куки, у которых пул непуст) различается вдвое, и обученная зависимость просто не та,
которую применяют.

Альтернатива: для обучающей строки брать **все размеченные дни, кроме её собственного**. Своя
метка по-прежнему исключена — а именно она и была бы утечкой. Зрелость пула при этом выравнивается,
и обучающий признак начинает означать то же, что инференсный. Законность опирается ровно на
предыдущий замер: раз сигнал не зависит от направления и величины сдвига во времени, пул «вокруг»
статистически эквивалентен пулу «до».

Сравниваю три варианта на фолде «неделя → неделя»: обучение 6–12 апреля, валидация 13–19, пул для
валидации обрезан 13-м числом. У этого фолда 408 ботов против примерно 120 на двухдневных фолдах
раздела 4 — а при таком объёме шум метрики уже не съедает разницу в 0.03.

In [30]:
%%time
LONG_FOLDS = ['2026-04-13', '2026-04-15', '2026-04-17']


def split(cut):
    """Длинный фолд: всё до cut — обучение, всё от cut до конца трейна — валидация."""
    return WS >= np.datetime64(cut), WS < np.datetime64(cut)


def rep_blocks(cut, mode):
    """Репутация для обучающей и валидационной части фолда. Пул — только из обучающей части."""
    va, tr = split(cut)
    ma, mv = train[tr], train[va]
    ea = ev_tr[ev_tr.cookie_id.isin(ma.cookie_id)]
    evv = ev_tr[ev_tr.cookie_id.isin(mv.cookie_id)]
    pa, pv = item_pairs(ea, ma), item_pairs(evv, mv)
    dc = day_counts(pa, ma, y[tr])
    builder = {'lodo': rep_lodo, 'past': rep_past}[mode]
    return builder(pa, dc, ma.cookie_id), rep_asof(pv, dc, cut, mv.cookie_id)


def fit_single(cut, extra_a=None, extra_v=None):
    va, tr = split(cut)
    cols = FEATURES + (list(extra_a.columns) if extra_a is not None else [])
    Xa = X_tr.loc[tr, FEATURES].reset_index(drop=True)
    Xv = X_tr.loc[va, FEATURES].reset_index(drop=True)
    if extra_a is not None:
        Xa, Xv = pd.concat([Xa, extra_a], axis=1), pd.concat([Xv, extra_v], axis=1)
    m = HistGradientBoostingClassifier(**PARAMS).fit(Xa[cols], y[tr])
    return precision_at_recall(y[va], m.predict_proba(Xv[cols])[:, 1])


cut = LONG_FOLDS[0]
A_lodo, V_lodo = rep_blocks(cut, 'lodo')
A_past, V_past = rep_blocks(cut, 'past')
pd.DataFrame([
    {'пул для обучающих строк': 'нет (база)', 'покрытие на обучении': None,
     'покрытие на валидации': None, 'P@R0.7': round(fit_single(cut), 4)},
    {'пул для обучающих строк': 'только прошлые дни', 'покрытие на обучении': round(A_past.ri_cov.mean(), 3),
     'покрытие на валидации': round(V_past.ri_cov.mean(), 3),
     'P@R0.7': round(fit_single(cut, A_past, V_past), 4)},
    {'пул для обучающих строк': 'все дни, кроме своего', 'покрытие на обучении': round(A_lodo.ri_cov.mean(), 3),
     'покрытие на валидации': round(V_lodo.ri_cov.mean(), 3),
     'P@R0.7': round(fit_single(cut, A_lodo, V_lodo), 4)},
])

CPU times: total: 47.3 s
Wall time: 12.7 s


,пул для обучающих строк,покрытие на обучении,покрытие на валидации,P@R0.7
0,нет (база),NaN,NaN,0.8195
1,только прошлые дни,0.314,0.62,0.8266
2,"все дни, кроме своего",0.555,0.62,0.8567


Числа: база 0.820, «только прошлое» 0.827, «все дни, кроме своего» 0.857. Покрытие на обучении при
этом поднимается с 0.31 до 0.56 против 0.62 на валидации — выигрыш ровно там, где устранено
расхождение.

Три сотых между двумя версиями одного и того же признака — больше, чем давала половина проверенных
мной гипотез. Это стоит запомнить отдельно от задачи: **когда признак зависит от объёма накопленной
истории, разница в этом объёме между обучением и инференсом сама становится источником ошибки**, и
лечится она выравниванием, а не регуляризацией.

Осторожный вариант «только прошлое» остаётся в коде. Если трактовать требование «признаки доступны
на момент `window_end_ts`» максимально строго — так, что и обучающая выборка не вправе смотреть
вперёд, — надо брать его: он даёт меньше, но причинен в обе стороны.

### 11.3 Абляция на трёх длинных фолдах

Один фолд ничего не доказывает. Повторяю замер на трёх, сдвигая границу: 13, 15 и 17 апреля.
Валидация у них вложенная, поэтому фолды не независимы — но обучающие выборки разные, и
согласованность результата всё же о чём-то говорит.

Заодно здесь же пересматриваю вывод раздела 10 про свёртку. Там медиана рангов выигрывала на
двухдневных фолдах; проверю, держится ли это, когда фолды длиннее и признаки сильнее.

In [31]:
%%time
def fit_bag(cut, extra_a=None, extra_v=None, n_bag=8, seed=SEED):
    """Полная модель плюс бэггинг. Возвращает нормированные ранги для всех вариантов свёртки."""
    va, tr = split(cut)
    cols = FEATURES + (list(extra_a.columns) if extra_a is not None else [])
    Xa = X_tr.loc[tr, FEATURES].reset_index(drop=True)
    Xv = X_tr.loc[va, FEATURES].reset_index(drop=True)
    if extra_a is not None:
        Xa, Xv = pd.concat([Xa, extra_a], axis=1), pd.concat([Xv, extra_v], axis=1)
    Xa, Xv, ya = Xa[cols], Xv[cols], y[tr]
    p_full = HistGradientBoostingClassifier(**PARAMS).fit(Xa, ya).predict_proba(Xv)[:, 1]
    rng = np.random.default_rng(seed)
    P = []
    for i in range(n_bag):
        idx = rng.choice(len(Xa), len(Xa), replace=True)
        mm = HistGradientBoostingClassifier(**{**PARAMS, 'random_state': 100 + i})
        P.append(mm.fit(Xa.iloc[idx], ya[idx]).predict_proba(Xv)[:, 1])
    n = len(Xv)
    R = np.array([rankdata(p) for p in P]) / n
    rf = rankdata(p_full) / n
    return {'одна модель': p_full, 'бэггинг, медиана': np.median(R, 0), 'бэггинг, среднее': R.mean(0),
            'смесь 0.7/0.3': 0.7 * rf + 0.3 * R.mean(0)}


rows = []
for cut in LONG_FOLDS:
    va, _ = split(cut)
    A, V = rep_blocks(cut, 'lodo')
    for name, extra in [('база, 99 признаков', None), ('+ репутация, 112', (A, V))]:
        sc = fit_bag(cut, *(extra if extra else (None, None)))
        rows.append({'фолд': cut, 'ботов': int(y[va].sum()), 'набор': name,
                     **{k: round(precision_at_recall(y[va], v), 4) for k, v in sc.items()}})
abl = pd.DataFrame(rows)
display(abl)
abl.groupby('набор', sort=False).mean(numeric_only=True).drop(columns='ботов').round(4)

,фолд,ботов,набор,одна модель,"бэггинг, медиана","бэггинг, среднее",смесь 0.7/0.3
0,2026-04-13,408,"база, 99 признаков",0.8195,0.7967,0.8338,0.8218
1,2026-04-13,408,"+ репутация, 112",0.8567,0.8614,0.8750,0.8800
2,2026-04-15,285,"база, 99 признаков",0.8000,0.8097,0.7882,0.7843
3,2026-04-15,285,"+ репутация, 112",0.8299,0.8439,0.8403,0.8264
4,2026-04-17,160,"база, 99 признаков",0.7943,0.7958,0.8058,0.7972
5,2026-04-17,160,"+ репутация, 112",0.8496,0.8485,0.8485,0.8561


CPU times: total: 14min 43s
Wall time: 3min 48s


,одна модель,"бэггинг, медиана","бэггинг, среднее",смесь 0.7/0.3
набор,,,,
"база, 99 признаков",0.8046,0.8007,0.8093,0.8011
"+ репутация, 112",0.8454,0.8513,0.8546,0.8542


Репутация выигрывает на всех трёх фолдах и при любой свёртке. На фолде 13 апреля смесь даёт 0.880
против 0.822 без репутации, одиночная модель — 0.857 против 0.820. Такой разрыв не списать на шум
даже по осторожному правилу раздела 4.

Со свёрткой сложнее, и разбору этого посвящён следующий подраздел. Пока видно только, что картина
раздела 10 не воспроизводится: там медиана рангов уверенно выигрывала у среднего, здесь на базовом
наборе она даёт 0.797 против 0.834, то есть проигрывает ему же.

### 11.4 Сколько на самом деле стоит одно число на фолде

Собирая этот раздел, я поймал себя на ошибке, которую стоит разобрать, потому что она обесценивает
половину сравнений в ноутбуке.

Сначала я измерял репутацию с округлённой базовой долей $p_0 = 0.0810$, потом переписал код
аккуратнее и поставил $p_0 = \bar{y} = 0.081057$. Это константа сглаживания; её третий знак не несёт
никакого смысла, признаки сдвигаются на $5 \cdot 10^{-5}$. Метрика на фолде при этом поехала с 0.885
на 0.864.

Отдельно важно, что обычная проверка на устойчивость этого не ловит.
`HistGradientBoostingClassifier` с `early_stopping=False` детерминирован: пять разных `random_state`
дают **бит в бит одинаковый** результат. Разброса по seed нет вообще, и соблазнительно счесть
замер точным. На деле поверхность метрики кусочно-постоянна и изрезана: P@R меняется только когда
куки меняются местами в рейтинге возле порога, а порог здесь отсекает около 340 куки, так что
перестановка девяти из них — это уже 0.02.

Поэтому варьирую не seed, а то, что **содержательно произвольно**: $p_0$ и силу сглаживания
$\alpha$. Каждая комбинация одинаково законна, и разброс по ним — честная оценка того, чего стоит
одно число.

In [32]:
%%time
cut = LONG_FOLDS[0]
va, tr = split(cut)
ma, mv = train[tr], train[va]
ea = ev_tr[ev_tr.cookie_id.isin(ma.cookie_id)]
evv = ev_tr[ev_tr.cookie_id.isin(mv.cookie_id)]
pa, pv = item_pairs(ea, ma), item_pairs(evv, mv)

rows = []
for p0, alpha in [(0.0810, 5.0), (0.081057, 5.0), (0.0815, 5.0), (0.0810, 4.0), (0.0810, 6.0)]:
    globals()['P0_RATE'], globals()['ALPHA'] = p0, alpha      # обе константы произвольны
    dc = day_counts(pa, ma, y[tr])
    A, V = rep_lodo(pa, dc, ma.cookie_id), rep_asof(pv, dc, cut, mv.cookie_id)
    sc = fit_bag(cut, A, V)
    rows.append({'p0': p0, 'alpha': alpha,
                 **{k: round(precision_at_recall(y[va], v), 4) for k, v in sc.items()}})
globals()['P0_RATE'], globals()['ALPHA'] = float(y.mean()), 5.0

sens = pd.DataFrame(rows)
display(sens)
pd.DataFrame({'медиана': sens.iloc[:, 2:].median(), 'разброс': sens.iloc[:, 2:].max() - sens.iloc[:, 2:].min()}).round(4)

,p0,alpha,одна модель,"бэггинг, медиана","бэггинг, среднее",смесь 0.7/0.3
0,0.081000,5.0,0.8854,0.8563,0.8720,0.8966
1,0.081057,5.0,0.8640,0.8563,0.8645,0.8773
2,0.081500,5.0,0.8827,0.8593,0.8619,0.8882
3,0.081000,4.0,0.8773,0.8567,0.8667,0.8746
4,0.081000,6.0,0.8773,0.8571,0.8645,0.8910


CPU times: total: 11min 52s
Wall time: 3min 1s


,медиана,разброс
одна модель,0.8773,0.0214
"бэггинг, медиана",0.8567,0.0030
"бэггинг, среднее",0.8645,0.0101
смесь 0.7/0.3,0.8882,0.0220


Ширина разброса по пяти одинаково законным константам — порядка 0.02 для одиночной модели. Это и
есть цена одного числа на фолде, и она вдвое больше, чем разница, по которой я в разделе 10 решал
судьбу календарных признаков.

Отсюда два вывода, и второй мне пришлось принять против собственного первоначального ответа.

**Про бэггинг.** Промежуточный вывод «с сильными признаками бэггинг больше не нужен, одиночная
модель выигрывает» был артефактом: 0.885 оказались удачным попаданием в изрезанную поверхность, а
не свойством одиночной модели. Под вариацией констант одиночная модель гуляет шире остальных
вариантов — она и обязана, потому что усреднение по бутстрэпу как раз сглаживает эту изрезанность.
Смесь 0.7 на полную модель и 0.3 на бэггинг из 24 держится лучше всех: она выигрывает и на фолде 13,
и под вариацией констант, и не проигрывает одиночной на фолдах 15 и 17. Её и беру в финал.

**Про правило из раздела 4.** Формулировка «разницу меньше 0.03 одиночная модель на пяти фолдах не
различает» верна, но не по той причине, которую я в неё вкладывал. Я думал про дисперсию по
обучающей выборке — а основной вклад даёт чувствительность к произвольным мелочам конструкции
признака, которая не видна ни по seed, ни по фолдам, если менять только их. Практический вывод:
разброс надо мерить, варьируя произвольные решения внутри пайплайна, а не только случайность
модели.

### 11.5 Что вокруг репутации проверено и отброшено

Три расширения, которые напрашиваются, и ни одно не прошло.


In [33]:
%%time
cut = LONG_FOLDS[0]
va, tr = split(cut)
ma, mv = train[tr], train[va]
ea = ev_tr[ev_tr.cookie_id.isin(ma.cookie_id)]
evv = ev_tr[ev_tr.cookie_id.isin(mv.cookie_id)]


def as_key(ev, key):
    """Подменить ключ: тот же код работает для запросов и user-agent."""
    return ev.drop(columns=['item_id']).rename(columns={key: 'item_id'}) if key != 'item_id' else ev


def rep_for_key(key, prefix):
    """Та же конструкция репутации, но по другому ключу."""
    pa, pv = item_pairs(as_key(ea, key), ma), item_pairs(as_key(evv, key), mv)
    dc = day_counts(pa, ma, y[tr])
    A = rep_lodo(pa, dc, ma.cookie_id).add_prefix(prefix)
    V = rep_asof(pv, dc, cut, mv.cookie_id).add_prefix(prefix)
    return A, V


def pool_type_bots(pairs_lab, y_lab, meta_lab, tau=0.6, min_items=3):
    """«Пуловые» боты: ходят по общему с другими ботами набору объявлений."""
    ys = pd.Series(np.asarray(y_lab), index=meta_lab.cookie_id)
    b = pairs_lab[pairs_lab.cookie_id.map(ys) == 1]
    nb = b.groupby('item_id', observed=True).cookie_id.nunique()
    g = b.assign(sh=(b.item_id.map(nb) >= 2).astype(float)).groupby('cookie_id').sh.agg(['mean', 'size'])
    return set(g[(g['mean'] >= tau) & (g['size'] >= min_items)].index)


A0, V0 = rep_blocks(cut, 'lodo')
pa0 = item_pairs(ea, ma)
keep = pool_type_bots(pa0, y[tr], ma)
y_clean = np.array([1 if c in keep else 0 for c in ma.cookie_id])
dc_clean = day_counts(pa0, ma, y_clean)
A_cl = rep_lodo(pa0, dc_clean, ma.cookie_id).add_prefix('cl_')
V_cl = rep_asof(item_pairs(evv, mv), dc_clean, cut, mv.cookie_id).add_prefix('cl_')
A_q, V_q = rep_for_key('search_query', 'q_')
A_u, V_u = rep_for_key('user_agent', 'u_')

print(f'«пуловых» ботов {len(keep)} из {int(y[tr].sum())}')
pd.DataFrame([
    {'вариант': 'репутация объявлений', 'P@R0.7': round(fit_single(cut, A0, V0), 4)},
    {'вариант': '+ репутация запросов',
     'P@R0.7': round(fit_single(cut, pd.concat([A0, A_q], axis=1), pd.concat([V0, V_q], axis=1)), 4)},
    {'вариант': '+ репутация user-agent',
     'P@R0.7': round(fit_single(cut, pd.concat([A0, A_u], axis=1), pd.concat([V0, V_u], axis=1)), 4)},
    {'вариант': '+ очищенный пул',
     'P@R0.7': round(fit_single(cut, pd.concat([A0, A_cl], axis=1), pd.concat([V0, V_cl], axis=1)), 4)},
])

«пуловых» ботов 231 из 491
CPU times: total: 1min 11s
Wall time: 18.8 s


,вариант,P@R0.7
0,репутация объявлений,0.8567
1,+ репутация запросов,0.8537
2,+ репутация user-agent,0.8869
3,+ очищенный пул,0.8679


**Репутация поисковых запросов.** Логика та же, что для объявлений, результат хуже — на фолде
13 апреля падение примерно на 0.03. Запросов, категорий и локаций мало, их бот-ставка — уже
известная модели информация, перепакованная в тринадцать новых колонок.

**Репутация сырого user-agent** не даёт ничего: на трёх длинных фолдах 0.8610 против 0.8605 без неё.
Различных UA в корпусе 148; всё, что этот признак несёт, уже вытащено группой `ua_*` и счётчиком
кук на том же агенте из `cr_*`.

**Очищенный пул как набор агрегатов.** Идея: часть ботов ходит по разрозненным объявлениям, как
люди, и засоряет статистику, поэтому пул строится лишь по «пуловым» ботам. Те же тринадцать
агрегатов по такому пулу проигрывают обычному: 0.851 против 0.861. Этот вывод оказался неполным —
в 11.6 видно, что очищенный пул работает, но только через накопительный LLR, которого здесь нет.

### 11.6 Сверка с решением nefedovdima и накопительный LLR

У одного из участников, [nefedovdima/avito-bot-detection](https://github.com/nefedovdima/avito-bot-detection),
на скрытом тесте 0.90251. Его решение открыто, и после закрытия контеста я сверил подходы на одном
стенде: те же 99 поведенческих признаков, фолд 13 апреля, одна модель.

| набор | признаков | P@R0.7 |
| --- | --- | --- |
| база | 99 | 0.8195 |
| + его пул: 7 дней, только прошлое | 103 | 0.8290 |
| + его итоговая связка: пул и очищенный пул | 108 | 0.8363 |
| + мои 13 | 112 | 0.8567 |
| + мои 13 и его очищенный пул | 117 | 0.8773 |

Числа получены на его конструкции, перенесённой на мой стенд; его код в ноутбук не включён, поэтому
таблица здесь не пересчитывается.

Его пул на этих данных слабее моего — 0.836 против 0.857, и разница ровно та, что разобрана в 11.2:
«только прошлое» против «все дни, кроме своего». Но один его элемент добавляется поверх моих
признаков, и остальные колонки его очищенного пула сверх него ничего не дают. Это **накопительный
логарифм отношения правдоподобий**:

$$\mathrm{LLR} = m \log\frac{p_b}{p_h} + (k-m)\log\frac{1-p_b}{1-p_h},$$

где $k$ — число объявлений куки, $m$ — сколько из них попало в очищенный пул, $p_b$ и $p_h$ — доли
попаданий у ботов и у людей, оценённые на истории. Это наивный Байес: попадания считаются
независимыми испытаниями, и свидетельства складываются.

Мой LLR в 13 признаках поитемный и усредняется по объявлениям куки — среднее не растёт с числом
свидетельств. Кука с тремя объявлениями, два из которых в пуле, и кука с шестьюдесятью, из которых
сорок в пуле, получают у меня близкие значения; у него — различающиеся на порядок, как и должно
быть, если сорок совпадений действительно убедительнее двух. Промахи при этом штрафуют: длинная серия
«не попал» работает в пользу человека. Это объясняет провал очищенного пула в 11.5 — отбор
«пуловых» ботов я воспроизвёл, а механизм накопления, который и несёт сигнал, нет.

Остаётся вопрос, в какой конструкции истории считать этот признак, и ответ не тот, что для тринадцати.

In [34]:
%%time
LABELS = pd.Series(y, index=train.cookie_id)
PAIRS_ALL_TR = item_pairs(ev_tr, train)


def cp_lodo(pairs, meta, labels):
    """Тот же признак, но история — все размеченные дни, кроме своего, как у 13 признаков."""
    ids = pd.Index(meta.cookie_id)
    day = meta.set_index('cookie_id').window_start_ts.dt.normalize()
    out = []
    for d, cur in day.groupby(day).groups.items():
        hist = ids[(day.reindex(ids) != d).to_numpy()]
        out.append(cp_llr(pairs, pd.Index(cur), *cp_fit(pairs, hist, labels)))
    return pd.concat(out).reindex(ids).to_numpy()


rows = []
for cut in LONG_FOLDS:
    va, tr = split(cut)
    ma, mv = train[tr], train[va]
    A, V = rep_blocks(cut, 'lodo')
    A_p, V_p = A.copy(), V.copy()
    A_p['cp_llr'] = cp_train(PAIRS_ALL_TR, ma, LABELS)
    V_p['cp_llr'] = cp_infer(PAIRS_ALL_TR, mv, ma, LABELS, cut)
    A_l, V_l = A.copy(), V.copy()
    A_l['cp_llr'] = cp_lodo(PAIRS_ALL_TR, ma, LABELS)
    V_l['cp_llr'] = cp_llr(PAIRS_ALL_TR, pd.Index(mv.cookie_id),
                           *cp_fit(PAIRS_ALL_TR, ma.cookie_id, LABELS)).to_numpy()
    rows.append({'фолд': cut, 'мои 13': round(fit_single(cut, A, V), 4),
                 '+ cp_llr, только прошлое': round(fit_single(cut, A_p, V_p), 4),
                 '+ cp_llr, все дни кроме своего': round(fit_single(cut, A_l, V_l), 4)})
cpt = pd.DataFrame(rows)
display(cpt)
cpt.drop(columns='фолд').mean().round(4)

,фолд,мои 13,"+ cp_llr, только прошлое","+ cp_llr, все дни кроме своего"
0,2026-04-13,0.8567,0.8800,0.8910
1,2026-04-15,0.8299,0.8299,0.8368
2,2026-04-17,0.8496,0.8507,0.8235


CPU times: total: 2min 46s
Wall time: 44 s


мои 13                            0.8454
+ cp_llr, только прошлое          0.8535
+ cp_llr, все дни кроме своего    0.8504
dtype: float64

В конструкции «только прошлое» накопительный признак не проигрывает ни на одном фолде; в
конструкции «все дни, кроме своего» на двух фолдах выигрывает больше, а на фолде 17 апреля теряет
0.026.

Почему конструкция, выигравшая для тринадцати признаков, здесь ненадёжна. Тринадцать — средние по
объявлениям куки, а среднее нечувствительно к числу слагаемых: небольшое расхождение между
обучающим и инференсным пулом даёт в нём такое же небольшое смещение. Накопительный LLR — сумма.
Каждое попадание добавляет $\log(p_b/p_h) \approx 2$, и систематическое расхождение в частоте
попаданий, ничтожное на одно объявление, умножается на $k$. Сигнал интегрируется вместе со
смещением, и у кук с сотней объявлений смещение выходит на первый план. Пул «вокруг» строки содержит
будущих ботов того же сервиса: в среднем это безвредно, в сумме — нет.

Отсюда правило, уточняющее 11.2: **чем сильнее статистика накапливает свидетельства, тем точнее её
обучающая конструкция должна повторять инференсную**. Для средних можно выравнивать зрелость пула,
для сумм нельзя. Поэтому `cp_llr` считается так же, как у автора идеи: история строго в прошлом,
окно 7 дней, на тесте — одна история, замороженная на 20 апреля.

Средний прирост около 0.008 лежит внутри разброса из 11.4, и в одиночку я бы на него не опирался.
У автора тот же элемент дал +0.007 на платформе — тоже в пределах его шума, но того же знака. Беру
признак потому, что он не проигрывает ни на одном фолде и у него ясный механизм.

## 12. Финальная модель и `submission.csv`

In [35]:
%%time
N_BAG = 24
HORIZON = '2026-04-20'          # первый день теста: пул меток целиком в прошлом относительно него
LABELS = pd.Series(y, index=train.cookie_id)

PAIRS_TR, PAIRS_TE = item_pairs(ev_tr, train), item_pairs(ev_te, test)
PAIRS_ALL = pd.concat([PAIRS_TR, PAIRS_TE], ignore_index=True)
DC = day_counts(PAIRS_TR, train, y)

REP_TR = rep_lodo(PAIRS_TR, DC, train.cookie_id)                 # средние: все дни, кроме своего
REP_TE = rep_asof(PAIRS_TE, DC, HORIZON, test.cookie_id)
REP_TR['cp_llr'] = cp_train(PAIRS_ALL, train, LABELS)            # сумма: только прошлое, 7 дней
REP_TE['cp_llr'] = cp_infer(PAIRS_ALL, test, train, LABELS, HORIZON)

COLS = FEATURES + REP + ['cp_llr']
XA = pd.concat([X_tr[FEATURES], REP_TR], axis=1)[COLS]
XT = pd.concat([X_te[FEATURES], REP_TE], axis=1)[COLS]
print('матрицы:', XA.shape, XT.shape)
print('покрытие репутации: обучение %.3f | тест %.3f' % (REP_TR.ri_cov.mean(), REP_TE.ri_cov.mean()))


def final_score(seed=SEED, n_bag=N_BAG):
    """Единственный путь вычисления score — используется и для сабмита, и для проверки."""
    p_full = HistGradientBoostingClassifier(**PARAMS).fit(XA, y).predict_proba(XT)[:, 1]
    rng = np.random.default_rng(seed)
    P = []
    for i in range(n_bag):
        idx = rng.choice(len(XA), len(XA), replace=True)
        m = HistGradientBoostingClassifier(**{**PARAMS, 'random_state': 100 + i})
        P.append(m.fit(XA.iloc[idx], y[idx]).predict_proba(XT)[:, 1])
    P = np.array(P)
    n = len(XT)
    bag_rank = (np.array([rankdata(p) for p in P]) / n).mean(0)
    s = 0.7 * rankdata(p_full) / n + 0.3 * bag_rank + 0.001 * P.mean(0)
    return (s - s.min()) / (s.max() - s.min())


sub = pd.DataFrame({'cookie_id': X_te.cookie_id, 'score': final_score()})
sub.to_csv('submission.csv', index=False)
sub.head()

матрицы: (11091, 113) (4909, 113)
покрытие репутации: обучение 0.794 | тест 0.818
CPU times: total: 8min 5s
Wall time: 2min 3s


,cookie_id,score
0,ck_315fb710a0e371e7,0.033484
1,ck_a76ee3b3e3e522fd,0.900270
2,ck_94c9a4d382689e82,0.646919
3,ck_8eaf9509ad9462a0,0.102961
4,ck_9a88a5a989cb5bc6,0.843942


In [36]:
chk = pd.read_csv('submission.csv')
assert list(chk.columns) == ['cookie_id', 'score'],      'ровно две колонки в нужном порядке'
assert len(chk) == len(test),                            f'ожидалось {len(test)} строк'
assert chk.cookie_id.is_unique,                          'дубликаты cookie_id'
assert set(chk.cookie_id) == set(test.cookie_id),        'набор cookie_id не совпадает с test.csv'
assert chk.score.notna().all(),                          'пропуски в score'
assert chk.score.between(0, 1).all(),                    'score вне [0, 1]'

print('строк:', len(chk))
print('уникальных score:', chk.score.nunique(), f'({chk.score.nunique() / len(chk):.1%})')
print('score: min %.4f | median %.4f | max %.4f' % (chk.score.min(), chk.score.median(), chk.score.max()))
print('все проверки пройдены')

строк: 4909
уникальных score: 4909 (100.0%)
score: min 0.0000 | median 0.4902 | max 1.0000
все проверки пройдены


In [37]:
%%time
# Воспроизводимость: тот же код, тот же seed — файл должен совпасть побайтово.
pd.DataFrame({'cookie_id': X_te.cookie_id, 'score': final_score()}).to_csv('_check.csv', index=False)
h = [hashlib.md5(open(f, 'rb').read()).hexdigest() for f in ('submission.csv', '_check.csv')]
print(h[0], '|', h[1], '| идентичны:', h[0] == h[1])

import os; os.remove('_check.csv')

e9b96b08517c14edcaa81ca7e467471f | e9b96b08517c14edcaa81ca7e467471f | идентичны: True
CPU times: total: 7min 59s
Wall time: 2min


## 13. Итоги

| шаг | пять коротких окон | три длинных фолда, одна модель |
| --- | --- | --- |
| константа | 0.094 | — |
| baseline из quickstart (объём активности) | 0.099 | — |
| 99 поведенческих признаков | 0.827 (на тесте 0.839) | 0.805 |
| + репутация объявлений, 13 средних | 0.849 | 0.845 |
| + накопительный LLR | — | 0.854 |

Короткие окна — исходная схема раздела 4, для неё единственной известна связь с платформой. Длинные
фолды — раздел 11, у них больше ботов, но вдвое меньше обучающих данных.

**Прирост от репутации реален, его размер — нет.** Он положителен в девяти замерах из десяти по двум
схемам, но на длинных фолдах составляет +0.04…+0.05, а на коротких +0.02. Правдоподобное объяснение:
репутация частично замещает объём обучающей выборки — на половине данных поведенческая модель
недоучена и внешняя подсказка весит больше, — а финал учится на полном трейне. Проверить это нельзя:
фолд, у которого одновременно много ботов в валидации и полный обучающий набор, четырнадцать дней
данных не дают.

**Что оказалось главным.** Поведение внутри окна упирается в потолок около 0.80–0.83: половина того,
что отличает парсера от человека, внутри одного окна не видна вообще. Решающим стал переход к
информации о соседях — какие объявления кука делит с уже известными ботами.

**Что оказалось обманчивым.** Трижды за задачу вывод по слабому замеру оказывался неверным:
календарные признаки (раздел 10), медиана рангов (11.3) и одиночная модель вместо бэггинга (11.4).
Все три раза ошибка одна: решение принималось там, где разрешающая способность валидации меньше
измеряемого эффекта. Рабочий ответ — удлинять фолд и варьировать произвольные константы пайплайна, а
не только seed.

**Ограничения.** Признаки `cr_*` и репутация считаются по чужим кукам. Для инференса обе группы
причинны: на тестовых строках пул целиком в прошлом. Обучающие строки для тринадцати средних смотрят
на соседние дни в обе стороны — обоснование в 11.1–11.2, осторожная альтернатива там же. Потолок
задачи задан разметкой: положительный класс — трафик *известных* сервисов, отрицательный — трафик,
*отнесённый* к человеческому, и репутация упирается в этот потолок прямее остальных признаков:
она переносит на новую куку то, что известно о старых, включая систематические ошибки разметки.